# Colab T4 Benchmark — Tez Aşama 0

**Sürüm: PUBLIC-ARCHIVE-v6 — tez_datasets yolu zorunlu aktarılır.**

Bu notebook bilimsel sonuç üretmez. CIFAR-10 Google Drive'daki `MyDrive/tez_datasets` klasöründen okunur; benchmark bu yolu açık parametre olarak alır ve internet indirmesine düşmez.

**Önce:** `Runtime > Change runtime type > T4 GPU` seçin.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


## GitHub reposunu al

Repo public olduğu için token veya Colab Secret gerekmez. `git clone` kullanılmaz; notebook GitHub'ın public API/arşiv bağlantısından `main` dalının güncel commitini indirir ve kullanılan commit SHA'sını kaydeder.


In [ ]:
import io, os, shutil, tarfile, requests
from pathlib import Path

OWNER = 'erdemci07'
REPO = 'derin-ogrenme-egitim-dinamikleri'
BRANCH = 'main'

repo_dir = Path('/content/derin-ogrenme-egitim-dinamikleri')
extract_root = Path('/content/_tez_repo_extract')
shutil.rmtree(repo_dir, ignore_errors=True)
shutil.rmtree(extract_root, ignore_errors=True)
extract_root.mkdir(parents=True, exist_ok=True)

commit_url = f'https://api.github.com/repos/{OWNER}/{REPO}/commits/{BRANCH}'
commit_resp = requests.get(commit_url, timeout=60)
commit_resp.raise_for_status()
commit_sha = commit_resp.json()['sha']

archive_url = f'https://api.github.com/repos/{OWNER}/{REPO}/tarball/{commit_sha}'
archive_resp = requests.get(archive_url, timeout=120)
archive_resp.raise_for_status()

with tarfile.open(fileobj=io.BytesIO(archive_resp.content), mode='r:gz') as tar:
    tar.extractall(extract_root)

kokler = [p for p in extract_root.iterdir() if p.is_dir()]
if len(kokler) != 1:
    raise RuntimeError('GitHub arşivi beklenen klasör yapısında değil.')

shutil.move(str(kokler[0]), str(repo_dir))
os.chdir(repo_dir)
os.environ['TEZ_GIT_COMMIT'] = commit_sha

print('Repo hazır.')
print('Commit:', commit_sha)


## Sonuçların Google Drive'a kalıcı kaydı


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_KOK = Path('/content/drive/MyDrive')
DRIVE_VERI = DRIVE_KOK / 'tez_datasets'
DRIVE_CIKTI = DRIVE_KOK / 'tez_deney_sonuclari' / 'benchmark_colab_t4'

if not DRIVE_VERI.is_dir():
    raise FileNotFoundError(f'Google Drive veri klasörü bulunamadı: {DRIVE_VERI}')

DRIVE_CIKTI.mkdir(parents=True, exist_ok=True)
os.environ['TEZ_VERI_KLASORU'] = str(DRIVE_VERI)

print('Veri klasörü:', DRIVE_VERI)
print('Çıktı klasörü:', DRIVE_CIKTI)


## Colab bağımlılıkları

Colab'ın kendi CUDA destekli PyTorch kurulumu korunur; bu nedenle `requirements_colab.txt` içinde torch/torchvision yeniden kurulmaz.


In [ ]:
subprocess.run([
    'python', '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'
], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA kullanılabilir:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA aktif değil. Runtime ayarından T4 GPU seçin.')


## Benchmarkı çalıştır

1 warmup + 3 ölçüm epochu CNN-v1 ve CIFAR uyarlanmış ResNet-18 için çalıştırılır.


In [ ]:
import importlib
import kaynak_kod.veri_hazirlama as veri_hazirlama_modulu
import benchmark_colab_t4 as benchmark_modulu

# Aynı Colab oturumunda eski modüller bellekte kalmışsa güncel dosyaları yeniden yükle.
importlib.reload(veri_hazirlama_modulu)
importlib.reload(benchmark_modulu)

cikti = DRIVE_CIKTI / 'benchmark_t4.json'
print('Benchmark fonksiyonu başlatılıyor...', flush=True)
print('Kullanılacak Drive veri klasörü:', DRIVE_VERI, flush=True)

benchmark_modulu.ana([
    '--warmup-epoch', '1',
    '--olcum-epoch', '3',
    '--batch-size', '128',
    '--num-workers', '0',
    '--seed', '42',
    '--veri-klasoru', str(DRIVE_VERI),
    '--cikti', str(cikti),
])


## Özeti göster


In [ ]:
import json
veri = json.loads(cikti.read_text(encoding='utf-8'))
print('GPU:', veri['ortam']['gpu'])
print('Commit:', veri['ortam']['git_commit'])
for sonuc in veri['benchmark']:
    print(
        sonuc['model'],
        'ortalama epoch =', round(sonuc['mean_total_seconds_per_epoch'], 2), 'sn',
        '| tepe bellek =', round(sonuc['peak_gpu_memory_mb'], 0), 'MB'
    )
print('\nKaba kapasite tahminleri:')
for k, v in veri['kaba_kapasite_tahminleri'].items():
    if isinstance(v, (int, float)):
        print('-', k, '=', round(v, 2), 'GPU-saat')
print('\nDosya:', cikti)


## Bu aşamadan sonra

`benchmark_t4.json` dosyasındaki gerçek süreler kullanılarak 150, 300 ve maksimum 1.035 koşuluk planın uygulanabilirliği hesaplanacaktır. Bu notebook çıktıları tez bulgusu değildir.
